In [233]:
# Monthly Stock Price Aggregation & Technical Indicator Computation

# ==================================
# BLOCK 1: Imports and Configuration
# ==================================

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [234]:
# Configuration
INPUT_FILE = 'stock_data.csv'  
OUTPUT_DIR = 'output'
VIZ_OUTPUT_DIR = 'visualizations'
TICKERS = ['AAPL', 'AMD', 'AMZN', 'AVGO', 'CSCO', 'MSFT', 'NFLX', 'PEP', 'TMUS', 'TSLA']

print("• Stock Data Monthly Resampling & Visualization Pipeline: ")
print("=" * 57)

• Stock Data Monthly Resampling & Visualization Pipeline: 


In [235]:

# ==============================
# BLOCK 2: Data Loading Function
# ==============================
def load_stock_data(filepath):
    """
    Load stock data from CSV file and prepare for processing.
    
    Parameters:
    -----------
    filepath : str
        Path to the input CSV file
        
    Returns:
    --------
    pd.DataFrame
        Loaded and preprocessed dataframe
    """
    df = pd.read_csv(filepath)
    
    #convert date to datetime
    df['date'] = pd.to_datetime(df['date'])
    
    # Sort by ticker and date
    df = df.sort_values(['ticker', 'date']).reset_index(drop=True)
    
    print(f"• Data loaded successfully: ")
    print(f"- Total records: {len(df):,}")
    print(f"- Date range: {df['date'].min().date()} to {df['date'].max().date()}")
    print(f"- Unique tickers: {df['ticker'].nunique()}")
    
    return df


In [236]:

# ==========================================
# BLOCK 3: Monthly OHLC Aggregation Function
# ==========================================
def resample_to_monthly_ohlc(df, ticker):
    """
    Resample daily data to monthly OHLC format.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Daily stock data for a single ticker
    ticker : str
        Stock ticker symbol
        
    Returns:
    --------
    pd.DataFrame
        Monthly aggregated data with OHLC values
    """
    # Set date as index for resampling
    df_ticker = df[df['ticker'] == ticker].copy()
    df_ticker = df_ticker.set_index('date')
    
    #resample to monthly frequency
    monthly = pd.DataFrame()
    
    # Open: First day's open price
    monthly['open'] = df_ticker['open'].resample('MS').first()
    
    # High: Maximum high price in the month
    monthly['high'] = df_ticker['high'].resample('MS').max()
    
    # Low: Minimum low price in the month
    monthly['low'] = df_ticker['low'].resample('MS').min()
    
    # Close: Last day's close price
    monthly['close'] = df_ticker['close'].resample('MS').last()
    
    #volume: Sum of volumes
    monthly['volume'] = df_ticker['volume'].resample('MS').sum()
    
    #reset index to make date a column
    monthly = monthly.reset_index()
    monthly.columns = ['date', 'open', 'high', 'low', 'close', 'volume']
    
    return monthly


In [237]:
# ================================================
# BLOCK 4: Simple Moving Average (SMA) Calculation
# ================================================
def calculate_sma(series, window):
    """
    Calculate Simple Moving Average.
    
    Parameters:
    -----------
    series : pd.Series
        Price series (typically closing prices)
    window : int
        Number of periods for the moving average
        
    Returns:
    --------
    pd.Series
        SMA values
    """
    return series.rolling(window=window, min_periods=1).mean()


In [238]:

# =====================================================
# BLOCK 5: Exponential Moving Average (EMA) Calculation
# =====================================================
def calculate_ema(series, window):
    alpha = 2 / (window + 1)
    ema = series.copy()
    
    # First EMA value = SMA of first `window` periods -- stated in assignment
    ema.iloc[window - 1] = series.iloc[:window].mean()
    
    # Apply recursive EMA
    for i in range(window, len(series)):
        ema.iloc[i] = (
            (series.iloc[i] - ema.iloc[i - 1]) * alpha
            + ema.iloc[i - 1]
        )
    
    return ema



In [239]:

# ======================================
# BLOCK 6: Technical Indicators Pipeline
# ======================================
def add_technical_indicators(df):
    """
    Add all technical indicators (SMA and EMA) to the dataframe.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Monthly aggregated data
        
    Returns:
    --------
    pd.DataFrame
        Dataframe with technical indicators added
    """
    # Calculate SMAs based on monthly closing prices
    df['sma_10'] = calculate_sma(df['close'], 10)
    df['sma_20'] = calculate_sma(df['close'], 20)
    
    # Calculate EMAs based on monthly closing prices
    df['ema_10'] = calculate_ema(df['close'], 10)
    df['ema_20'] = calculate_ema(df['close'], 20)
    
    # Round to 2 decimal places for readability
    indicator_cols = ['sma_10', 'sma_20', 'ema_10', 'ema_20']
    df[indicator_cols] = df[indicator_cols].round(2)
    
    return df


In [240]:
# =============================
# BLOCK 7: File Export Function
# =============================
def export_ticker_data(df, ticker, output_dir):
    """
    Export processed data for a single ticker to CSV.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Processed monthly data with indicators
    ticker : str
        Stock ticker symbol
    output_dir : str or Path
        Directory to save the output file
    """
    # create o/p directory if it doesn't exist
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    #define o/p filename
    output_file = Path(output_dir) / f"result_{ticker}.csv"
    
    # Export to CSV
    df.to_csv(output_file, index=False)
    
    return output_file

In [241]:
# =================================
# BLOCK 8: Main Processing Pipeline
# =================================
def process_stock_data(input_file, tickers, output_dir):
    """
    Main pipeline to process all stock data.
    
    Parameters:
    -----------
    input_file : str
        Path to input CSV file
    tickers : list
        List of ticker symbols to process
    output_dir : str
        Directory to save output files
        
    Returns:
    --------
    dict
        Dictionary with ticker as key and processed dataframe as value
    """
    # Load data
    df = load_stock_data(input_file)
    
    print("\n• Processing tickers: ")
    print("-" * 46)
    
    results = {}
    
    for ticker in tickers:
        # Check if ticker exists in data
        if ticker not in df['ticker'].values:
            print(f"⚠️  {ticker}: Not found in dataset - Skipping")  # not found 
            continue
        
        #resample to monthly
        monthly_df = resample_to_monthly_ohlc(df, ticker)
        
        # add technical indicators
        monthly_df = add_technical_indicators(monthly_df)
        
        # Export to file
        output_file = export_ticker_data(monthly_df, ticker, output_dir)
        
        # Store in results
        results[ticker] = monthly_df
        
        print(f"☑️ {ticker}: Processed {len(monthly_df)} months → {output_file.name}")
    
    print("-" * 46)
    print(f"\nPipeline completed. \n{len(results)} files generated in :'{output_dir}/' folder.")
    
    return results

In [242]:
# =========================
# BLOCK 9: Execute Pipeline
# =========================
# Run the complete pipeline
results = process_stock_data(INPUT_FILE, TICKERS, OUTPUT_DIR)

• Data loaded successfully: 
- Total records: 5,030
- Date range: 2018-01-02 to 2019-12-31
- Unique tickers: 10

• Processing tickers: 
----------------------------------------------
☑️ AAPL: Processed 24 months → result_AAPL.csv
☑️ AMD: Processed 24 months → result_AMD.csv
☑️ AMZN: Processed 24 months → result_AMZN.csv
☑️ AVGO: Processed 24 months → result_AVGO.csv
☑️ CSCO: Processed 24 months → result_CSCO.csv
☑️ MSFT: Processed 24 months → result_MSFT.csv
☑️ NFLX: Processed 24 months → result_NFLX.csv
☑️ PEP: Processed 24 months → result_PEP.csv
☑️ TMUS: Processed 24 months → result_TMUS.csv
☑️ TSLA: Processed 24 months → result_TSLA.csv
----------------------------------------------

Pipeline completed. 
10 files generated in :'output/' folder.


In [243]:

# =================================
# BLOCK 10: Data Quality Validation
# =================================
def validate_results(results):
    """
    Validate the processed results for quality checks.
    
    Parameters:
    -----------
    results : dict
        Dictionary of processed dataframes
    """
    print("\n• Data Quality Validation: ")
    print("=" * 26)
    
    for ticker, df in results.items():
        print(f"\n{ticker}:")
        print(f"  · Number of months: {len(df)}")
        print(f"  · Date range: {df['date'].min().date()} to {df['date'].max().date()}")
        print(f"  · Missing values: {df.isnull().sum().sum()}")
        print(f"  · Price range: ${df['low'].min():.2f} - ${df['high'].max():.2f}")
        
        # Validate OHLC logic
        valid_ohlc = (df['low'] <= df['open']) & (df['open'] <= df['high']) & \
                     (df['low'] <= df['close']) & (df['close'] <= df['high'])
        
        if valid_ohlc.all():
            print(f"  · OHLC validation: All records valid ☑️")
        else:
            print(f"  · OHLC validation: ⚠️  {(~valid_ohlc).sum()} invalid records")

# Run validation
validate_results(results)


• Data Quality Validation: 

AAPL:
  · Number of months: 24
  · Date range: 2018-01-01 to 2019-12-01
  · Missing values: 0
  · Price range: $142.00 - $293.97
  · OHLC validation: All records valid ☑️

AMD:
  · Number of months: 24
  · Date range: 2018-01-01 to 2019-12-01
  · Missing values: 0
  · Price range: $9.04 - $47.31
  · OHLC validation: All records valid ☑️

AMZN:
  · Number of months: 24
  · Date range: 2018-01-01 to 2019-12-01
  · Missing values: 0
  · Price range: $1170.51 - $2050.50
  · OHLC validation: All records valid ☑️

AVGO:
  · Number of months: 24
  · Date range: 2018-01-01 to 2019-12-01
  · Missing values: 0
  · Price range: $197.46 - $331.20
  · OHLC validation: All records valid ☑️

CSCO:
  · Number of months: 24
  · Date range: 2018-01-01 to 2019-12-01
  · Missing values: 0
  · Price range: $37.35 - $58.26
  · OHLC validation: All records valid ☑️

MSFT:
  · Number of months: 24
  · Date range: 2018-01-01 to 2019-12-01
  · Missing values: 0
  · Price range: $83

In [244]:
# ============================
# BLOCK 11: Summary Statistics
# ============================
def generate_summary_statistics(results):
    """
    Generate summary statistics across all tickers.
    
    Parameters:
    -----------
    results : dict
        Dictionary of processed dataframes
        
    Returns:
    --------
    pd.DataFrame
        Summary statistics dataframe
    """
    summary_data = []
    
    for ticker, df in results.items():
        summary_data.append({
            'Ticker': ticker,
            'Months': len(df),
            'Avg_Close': df['close'].mean(),
            'Min_Price': df['low'].min(),
            'Max_Price': df['high'].max(),
            'Avg_Volume': df['volume'].mean(),
            'Final_SMA10': df['sma_10'].iloc[-1],
            'Final_SMA20': df['sma_20'].iloc[-1],
            'Final_EMA10': df['ema_10'].iloc[-1],
            'Final_EMA20': df['ema_20'].iloc[-1],
        })
    
    summary_df = pd.DataFrame(summary_data)
    
    print("\n• Summary Statistics Across All Tickers:")
    print("=" * 40)
    print(summary_df.to_string(index=False))
    
    return summary_df

# Generate summary
if results:
    summary = generate_summary_statistics(results)


• Summary Statistics Across All Tickers:
Ticker  Months   Avg_Close   Min_Price   Max_Price   Avg_Volume  Final_SMA10  Final_SMA20  Final_EMA10  Final_EMA20
  AAPL      24  200.334166  142.000000  293.970001 6.501183e+08       221.90       206.47       232.60       212.65
   AMD      24   24.023333    9.040000   47.310001 1.581055e+09        32.08        26.49        33.09        26.79
  AMZN      24 1726.180832 1170.510010 2050.500000 9.969279e+07      1818.03      1772.58      1790.42      1740.23
  AVGO      24  263.874165  197.460007  331.200012 7.096354e+07       293.24       268.67       291.22       271.59
  CSCO      24   47.627500   37.349998   58.259998 4.739762e+08        50.91        48.48        48.87        47.59
  MSFT      24  117.138334   83.830002  159.550003 5.885000e+08       137.18       121.89       137.98       123.42
  NFLX      24  324.693751  195.419998  423.209991 2.024601e+08       324.77       331.16       315.86       320.32
   PEP      24  120.091668   9

In [245]:
# ==============================
# BLOCK 12: Final Report Summary
# ==============================
def generate_final_report(results, summary_df):
    """
    Generate a comprehensive final report.
    """
    print("\n" + "=" * 36)
    print("• FINAL REPORT - STOCK DATA ANALYSIS")
    print("=" * 36)
    
    print("\n1. DATA PROCESSING SUMMARY:")
    print(f"   • Total tickers processed: {len(results)}")
    print(f"   • Months per ticker: {len(list(results.values())[0]) if results else 0}")
    print(f"   • Output files generated: {len(results)}")
    print(f"   • Output directory: {OUTPUT_DIR}/")
    
    print("\n2. TECHNICAL INDICATORS CALCULATED:")
    print("   • SMA 10 (Simple Moving Average - 10 periods)")
    print("   • SMA 20 (Simple Moving Average - 20 periods)")
    print("   • EMA 10 (Exponential Moving Average - 10 periods)")
    print("   • EMA 20 (Exponential Moving Average - 20 periods)")
    
    print("\n6. FILES STRUCTURE:")
    print(f"   {OUTPUT_DIR}/")
    for ticker in sorted(results.keys()):
        print(f"   ├── result_{ticker}.csv")
    print(f"   {INPUT_FILE}/")
    print(f"   {'submission.ipynb'}/")
 
    print("\n" + "=" * 53)
    print("ANALYSIS COMPLETE - All files generated successfully!")
    print("=" * 53 + "\n")

# Generate final report
if results and 'summary' in locals():
    generate_final_report(results, summary)


print("=" * 42)
print("PIPELINE EXECUTION COMPLETED SUCCESSFULLY!")
print("=" * 42)


• FINAL REPORT - STOCK DATA ANALYSIS

1. DATA PROCESSING SUMMARY:
   • Total tickers processed: 10
   • Months per ticker: 24
   • Output files generated: 10
   • Output directory: output/

2. TECHNICAL INDICATORS CALCULATED:
   • SMA 10 (Simple Moving Average - 10 periods)
   • SMA 20 (Simple Moving Average - 20 periods)
   • EMA 10 (Exponential Moving Average - 10 periods)
   • EMA 20 (Exponential Moving Average - 20 periods)

6. FILES STRUCTURE:
   output/
   ├── result_AAPL.csv
   ├── result_AMD.csv
   ├── result_AMZN.csv
   ├── result_AVGO.csv
   ├── result_CSCO.csv
   ├── result_MSFT.csv
   ├── result_NFLX.csv
   ├── result_PEP.csv
   ├── result_TMUS.csv
   ├── result_TSLA.csv
   stock_data.csv/
   submission.ipynb/

ANALYSIS COMPLETE - All files generated successfully!

PIPELINE EXECUTION COMPLETED SUCCESSFULLY!
